# AlphaFold2 Ablation Study — Local Plotting (v3)

Copy of `plot_all_windows_v2.ipynb` with **complete parameter documentation** (see the *Parameter reference* cell below).
Same `combine_plots` pipeline, with every presentation control: custom colours & per-experiment opacity, legend
layout / mode (inline · separate file · none), custom tick grid, axis titles, per-element font sizes, and optional title.

No `uv` or shell commands needed — everything runs as pure Python.

In [ ]:
import os, sys

# Make sure we're in the project root (not the notebooks folder)
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Add the scripts folder to the Python path so we can import from it
scripts_dir = os.path.join(os.getcwd(), 'scripts')
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

print('Working directory:', os.getcwd())
print('Scripts dir on path:', scripts_dir)

In [ ]:
# --- CONFIGURE THESE PATHS ---

# Where your experiments are
result_path = r"C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output"
# Where your repository code and 'data' folder is
base_repo_path = r"C:\Users\franc\OneDrive - TUM\2 - Protein Pred\Code\alphafold2-ablation-study"

print('Input folder:', base_repo_path)
print('Output folder:', result_path)
print('Input exists:', os.path.isdir(base_repo_path))
print('Output exists:', os.path.isdir(result_path))

## Parameter reference — `combine_plots` / `plot_tm_score`

`combine_plots(data_files, ...)` merges several per-experiment CSVs into one scatter, then calls `plot_tm_score`.
**Every styling parameter below exists on both functions** (call `plot_tm_score` directly for a single CSV).
Parameters added after the original script are marked **(NEW)**.

### Files, protein & title
- **`data_files`** (list[str]) — CSVs to merge into one plot. *(combine_plots)*
- **`data_file`** (str) — a single CSV. *(plot_tm_score)*
- **`save_file_name`** (str) — output PNG file name.
- **`output_dir`** (str) — folder for the PNG (and, for `combine_plots`, the merged CSV).
- **`protein`** (str, default `None`) — needed only if a CSV holds more than one protein.
- **`title`** (str, default `None`) — overrides the auto title `"{protein} | Depth: {depths}"`.
- **`experiment_name`** (str, default `None`) — prepended to the auto title.
- **`show_title`** (bool, **NEW**, default `True`) — `True` keeps the title (auto or your `title`); `False` = **no title**.

### Data → visual encoding
- **`color_on`** (str, default `"nseq"`) — column mapped to point **colour**:
  - `"nseq"` (MSA depth) → depth **colour bar** (default);
  - a **text** column (e.g. `"experiment"`) → discrete **colour legend**;
  - a **numeric** non-`nseq` column (e.g. `"seed"`) → numeric colour bar.
- **`shape_on`** (str, default `None`) — column mapped to marker **shape** (discrete shape legend). Combinable with `color_on`.

### Colours **(NEW)**
- **`colors`** (list, default `None`) — point colours for a categorical colour legend (`color_on` = a text column).
  Any Matplotlib colour (hex, name, RGB tuple). **Aligned to the sorted unique category order** (see tip). Cycles if
  shorter than the number of categories. Defaults to the Okabe–Ito palette.
- Helper: `from plot_tmscore import get_okabe_ito_colors` → the 8 palette hex codes (**index 0 = black**), so you can
  mix palette + custom, e.g. `colors=[pal[1], "#000000"]`.

### Opacity
- **`opacity`** (float **or** list, default `1`) — point transparency (`0` = invisible … `1` = solid).
  - a single float → same for all points;
  - **(NEW)** a **list aligned to sorted categories** → **per-experiment** opacity — *only* when `color_on` is a text
    column (the discrete-colour path). One value per category, in sorted order. *(A list is not supported with
    `shape_on` or the colour-bar paths.)*

### Legend
- **`legend_title`** (str, default `None`) — legend heading text.
- **`legend_labels`** (list, default `None`) — text per legend entry, **aligned to sorted category order**.
- **`legend_layout`** (str, **NEW**, default `"auto"`) — `"auto"`/`"row"` = one horizontal row; `"column"` = stacked vertically.
- **`legend_mode`** (str, **NEW**, default `"inline"`) — where the discrete legend goes:
  - `"inline"` — on the plot (current behaviour);
  - `"separate"` — **not** on the plot; exported to a second file named like the PNG plus a `_legend` suffix
    (e.g. `combined_MCT1_v3.png` → `combined_MCT1_v3_legend.png`);
  - `"none"` — no legend at all.
  *(Affects the discrete colour/shape legend only; the `nseq` depth colour bar is unaffected.)*
- **`legend_title_fontsize`** (float, **NEW**, default `None`) — absolute pt for the legend **title**.
- **`legend_text_fontsize`** (float, **NEW**, default `None`) — absolute pt for the legend **entries**.

### Axes, ticks & guidelines
- **`limit_axis`** (bool, default `True`) — apply axis limits.
- **`axis_bounds`** (`[x_lo, x_hi, y_lo, y_hi]`, default `None`) — explicit limits (needs `limit_axis=True`; otherwise per-protein defaults).
- **`x_axis_title`** / **`y_axis_title`** (str, **NEW**, default `None`) — replace the auto `"Similarity to … (TM-score)"` labels.
- **`axis_title_fontsize`** (float, **NEW**, default `None`) — absolute pt for **both** axis titles.
- **`tick_anchor`** (float, **NEW**, default `None`) — a value the tick grid is aligned to (e.g. `1.0` or `0.95`).
- **`tick_size`** (float, **NEW**, default `None`) — tick spacing (e.g. `0.05`). With `tick_anchor`, ticks are placed at
  `anchor + k·size` for **every** integer `k` inside the axis range — **both axes, identical**. Both must be set;
  otherwise ~6 auto ticks are used.
- **`plot_guidelines`** (bool, default `True`) — dashed IF/OF (or active/inactive) reference lines.

### Fonts
- **`font_size`** (int, default `6`) — global base size. The title (`+5`), tick labels (`+3`), and any **unset** size
  below are derived from it as `font_size + N`.
- The **NEW** `*_fontsize` args — `axis_title_fontsize` (default `+1`), `legend_text_fontsize` (default `+2`),
  `legend_title_fontsize` (default `+3`) — are **absolute point sizes** that override that derivation for their
  element; leave them unset to keep the `font_size`-relative defaults.

> **Alignment tip — read this.** `legend_labels`, `colors`, and a per-experiment `opacity` list are all assigned to
> the point groups in the **sorted order of the `experiment` column values** (the full paths baked into each CSV,
> *not* the folder names and *not* your `experiments` list order). Confirm the order first:
>
> ```python
> import pandas as pd, numpy as np
> combined = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
> for i, c in enumerate(np.sort(combined["experiment"].unique())):
>     print(i, c)
> ```
>
> Then order the lists so index *i* matches category *i*. (For this dataset the sorted order is `depth_5120`,
> `query_mask_15`, `query_mask_15_Seed1`, `query_mask_15_Seed2` — note `query_mask_15` and `query_mask_15_Seed7`
> are the **same** base experiment, seed 7 just left unspecified.)

In [ ]:
# import os
# from plot_tmscore import combine_plots, get_okabe_ito_colors

# # --- experiments (folder names under plots/TM_Score) ---
# experiments = [
#     "depth_5120",
#     "query_mask_15_Seed1",
#     "query_mask_15_Seed2",
#     "query_mask_15_Seed7",
# ]

# protein_name = "MCT1"

# csv_files = [
#     os.path.join(result_path, "plots", "TM_Score", exp, f"{protein_name}.csv")
#     for exp in experiments
# ]

# # Okabe-Ito palette (colour-blind safe). Mix palette entries with your own hex colours.
# pal = get_okabe_ito_colors()

# combine_plots(
#     data_files=csv_files,
#     save_file_name=f"combined_{protein_name}_v2.png",
#     limit_axis=True,
#     axis_bounds=[0.7, 1.0, 0.7, 1.0],

#     # --- colour points by experiment (discrete colour legend) ---
#     color_on="experiment",
#     colors=[pal[1], pal[2], pal[3], "#000000"],       # SORTED-order aligned; mix palette + custom hex

#     # --- legend ---
#     legend_title="QM Seeds",
#     legend_labels=["No QM", "QM Seed 0", "QM Seed 1", "QM Seed 3"],  # sorted-order aligned
#     legend_layout="column",                           # "auto" | "row" | "column"
#     legend_title_fontsize=11,
#     legend_text_fontsize=9,

#     # --- custom tick grid (x & y): ticks at 0.95 +/- k*0.05 within the axis range ---
#     tick_anchor=0.95,
#     tick_size=0.05,

#     # --- axis titles ---
#     x_axis_title="TM-score to inward-facing state",
#     y_axis_title="TM-score to outward-facing state",
#     axis_title_fontsize=10,

#     output_dir=os.path.join(result_path, "plots", "TM_Score"),
# )

In [ ]:
import os
from plot_tmscore import combine_plots

experiments = [
    "depth_5120",
    "query_mask_15_Seed1",
    "query_mask_15_Seed2",
    "query_mask_15_Seed7",
]
protein_name = "MCT1"
csv_files = [
    os.path.join(result_path, "plots", "TM_Score", exp, f"{protein_name}.csv")
    for exp in experiments
]

combine_plots(
    data_files=csv_files,
    save_file_name=f"combined_{protein_name}_v3.png",
    limit_axis=True,
    axis_bounds=[0.7, 1.0, 0.7, 1.0],

    color_on="experiment",
    legend_title="QM Seeds",
    legend_labels=["No QM", "QM Seed 0", "QM Seed 1", "QM Seed 3"],
    opacity=[1.0, 1.0, 1.0, 1.0],   # same order as in experiments and colors

    # --- custom colours = the current standard Okabe–Ito assignment (by SORTED category) ---
    colors=[
        "#e69f00",  # cat 0: depth_5120            -> "No QM"     | Okabe–Ito[0] = black
        "#56b4e9",  # cat 1: query_mask_15 (Seed7) -> "QM Seed 0" | Okabe–Ito[1] = orange
        "#009e73",  # cat 2: query_mask_15_Seed1   -> "QM Seed 1" | Okabe–Ito[2] = sky blue
        "#cc79a7",  # cat 3: query_mask_15_Seed2   -> "QM Seed 3" | Okabe–Ito[3] = bluish green
    ],

    # --- axis titles ---
    x_axis_title="TM-Score: Pred vs IF Conf.",
    y_axis_title="TM-Score: Pred vs OF Conf.",

    # --- ticks: step 0.05, aligned so the top tick is 1.0 (grid steps down 1.0, 0.95, ...) ---
    tick_anchor=1.0,
    tick_size=0.05,

    # --- every configurable font size, set to the current defaults (font_size=6 base) ---
    font_size=6,                 # base size (title, tick labels) — current default
    axis_title_fontsize=9,       # default = font_size + 1
    legend_text_fontsize=9,      # default = font_size + 2
    legend_title_fontsize=9,     # default = font_size + 3
    # legend_layout="default"
    legend_layout="column",


    # --- title & legend output modes (NEW) ---
    show_title=False,          # True = current title; False = no title
    legend_mode="separate",     # "inline" = on plot | "separate" = export <name>_legend.png | "none" = no legend
    output_dir=os.path.join(result_path, "plots", "TM_Score"),
)


### Minimal / backward-compatible example

Every new parameter is optional. Omitting them reproduces the original behaviour
(depth colour bar or plain colour legend, auto axis labels, ~6 auto ticks, `font_size`-relative sizes).
The call below is the pre-v2 style — uncomment to run.

In [ ]:
# combine_plots(
#     data_files=csv_files,
#     save_file_name=f"combined_{protein_name}.png",
#     limit_axis=True,
#     axis_bounds=[0.7, 1.0, 0.7, 1.0],
#     color_on="experiment",
#     legend_title="QM Seeds",
#     legend_labels=["No QM", "QM Seed 0", "QM Seed 1", "QM Seed 3"],
#     output_dir=os.path.join(result_path, "plots", "TM_Score"),
# )